In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [2]:
PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
df = pd.read_csv(
    PROCESSED_DIR / "air_quality_clean.csv",
    parse_dates=['time'],
    index_col='time'
)

df.head()

,pm10,pm25,no2,so2,co,temperature,rain,wind_direction,wind_speed,humidity,pressure,dew_point
time,,,,,,,,,,,,
2022-09-01 00:00:00,53.1,37.1,29.3,5.0,765,20.6,1.2,243,0.8,95,870.2,19.8
2022-09-01 01:00:00,51.6,36.2,24.5,5.4,728,20.7,8.4,194,1.5,95,870.9,19.8
2022-09-01 02:00:00,50.6,35.5,18.8,5.8,679,20.9,9.5,225,1.5,94,871.7,19.9
2022-09-01 03:00:00,49.6,34.8,15.9,6.0,653,21.1,7.2,207,0.8,94,872.4,20.2
2022-09-01 04:00:00,47.1,33.1,13.9,6.1,629,21.4,2.6,45,1.0,93,873.0,20.3


In [4]:
print("Shape:", df.shape)
print("Start:", df.index.min())
print("End:", df.index.max())
print("Frequency:", pd.infer_freq(df.index))
print("Missing:", df.isna().sum().sum())

Shape: (23352, 12)
Start: 2022-09-01 00:00:00
End: 2025-04-30 23:00:00
Frequency: h
Missing: 0


In [5]:
features = df.copy()

In [6]:
#Calendar features for RQI
features['year'] = features.index.year

features['month'] = features.index.month
features['month_name'] = features.index.month_name()

features['quarter'] = features.index.quarter

features['day'] = features.index.day

features['day_of_year'] = features.index.dayofyear

features['day_of_week'] = features.index.dayofweek

features['day_name'] = features.index.day_name()

features['hour'] = features.index.hour

features['week_of_year'] = (
    features.index
    .isocalendar()
    .week
    .astype(int)
)

In [7]:
features[
    [
        'year',
        'month',
        'month_name',
        'day_of_week',
        'day_name',
        'hour'
    ]
].head()

,year,month,month_name,day_of_week,day_name,hour
time,,,,,,
2022-09-01 00:00:00,2022,9,September,3,Thursday,0
2022-09-01 01:00:00,2022,9,September,3,Thursday,1
2022-09-01 02:00:00,2022,9,September,3,Thursday,2
2022-09-01 03:00:00,2022,9,September,3,Thursday,3
2022-09-01 04:00:00,2022,9,September,3,Thursday,4


In [8]:
features['is_weekend'] = (
    features['day_of_week'] >= 5
).astype(int)

In [9]:
features['day_type'] = np.where(
    features['is_weekend'] == 1,
    'Weekend',
    'Weekday'
)

In [10]:
features[
    ['day_name', 'day_type']
].drop_duplicates().sort_values('day_name')

,day_name,day_type
time,,
2022-09-02,Friday,Weekday
2022-09-05,Monday,Weekday
2022-09-03,Saturday,Weekend
2022-09-04,Sunday,Weekend
2022-09-01,Thursday,Weekday
2022-09-06,Tuesday,Weekday
2022-09-07,Wednesday,Weekday


In [11]:
#Time-of-day categories
def get_time_of_day(hour):

    if 0 <= hour < 6:
        return 'Night'

    elif 6 <= hour < 12:
        return 'Morning'

    elif 12 <= hour < 18:
        return 'Afternoon'

    else:
        return 'Evening'


features['time_of_day'] = (
    features['hour']
    .apply(get_time_of_day)
)

In [12]:
features['time_of_day'].value_counts()

time_of_day
Night        5838
Morning      5838
Afternoon    5838
Evening      5838
Name: count, dtype: int64

## Nepal climatological seasons
Winter          December–February
Pre-Monsoon     March–May
Monsoon         June–September
Post-Monsoon    October–November

In [13]:
def get_season(month):

    if month in [12, 1, 2]:
        return 'Winter'

    elif month in [3, 4, 5]:
        return 'Pre-Monsoon'

    elif month in [6, 7, 8, 9]:
        return 'Monsoon'

    else:
        return 'Post-Monsoon'


features['season'] = (
    features['month']
    .apply(get_season)
)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [14]:
season_order = [
    'Winter',
    'Pre-Monsoon',
    'Monsoon',
    'Post-Monsoon'
]

features['season'] = pd.Categorical(
    features['season'],
    categories=season_order,
    ordered=True
)

In [15]:
features['season'].value_counts().sort_index()

season
Winter          6504
Pre-Monsoon     5880
Monsoon         6576
Post-Monsoon    4392
Name: count, dtype: int64

In [16]:
#timestamp note
print("Timezone information:", features.index.tz)

Timezone information: None


Hour-of-day features are extracted from the timestamps as supplied by the original data source. The source timezone should be verified before interpreting diurnal pollution patterns.

In [17]:
#Cyclical hour encoding
features['hour_sin'] = np.sin(
    2 * np.pi * features['hour'] / 24
)

features['hour_cos'] = np.cos(
    2 * np.pi * features['hour'] / 24
)

In [19]:
#cyclic day of week encoding
features['dow_sin'] = np.sin(
    2 * np.pi * features['day_of_week'] / 7
)

features['dow_cos'] = np.cos(
    2 * np.pi * features['day_of_week'] / 7
)

In [20]:
#Cyclical month encoding
features['month_sin'] = np.sin(
    2 * np.pi * (features['month'] - 1) / 12
)

features['month_cos'] = np.cos(
    2 * np.pi * (features['month'] - 1) / 12
)

In [21]:
#Day-of-year cyclical encoding
features['doy_sin'] = np.sin(
    2 * np.pi * features['day_of_year'] / 365.25
)

features['doy_cos'] = np.cos(
    2 * np.pi * features['day_of_year'] / 365.25
)

In [22]:
#Wind direction — circular encoding
wind_radians = np.deg2rad(
    features['wind_direction']
)

In [23]:
features['wind_dir_sin'] = np.sin(
    wind_radians
)

features['wind_dir_cos'] = np.cos(
    wind_radians
)

In [24]:
#wind direction
wind_labels = np.array([
    'N',
    'NE',
    'E',
    'SE',
    'S',
    'SW',
    'W',
    'NW'
])

wind_sector = (
    (
        features['wind_direction'] + 22.5
    ) // 45
).astype(int) % 8

features['wind_sector'] = (
    wind_labels[wind_sector]
)

In [25]:
features[
    ['wind_direction', 'wind_sector']
].head(20)

,wind_direction,wind_sector
time,,
2022-09-01 00:00:00,243,SW
2022-09-01 01:00:00,194,S
2022-09-01 02:00:00,225,SW
2022-09-01 03:00:00,207,SW
2022-09-01 04:00:00,45,NE
2022-09-01 05:00:00,324,NW
2022-09-01 06:00:00,328,NW
2022-09-01 07:00:00,328,NW
2022-09-01 08:00:00,79,E


In [26]:
wind_order = [
    'N',
    'NE',
    'E',
    'SE',
    'S',
    'SW',
    'W',
    'NW'
]

features['wind_sector'] = pd.Categorical(
    features['wind_sector'],
    categories=wind_order,
    ordered=True
)

In [27]:
#Rain/no-rain indicator
features['is_raining'] = (
    features['rain'] > 0
).astype(int)

In [28]:
features['rain_status'] = np.where(
    features['is_raining'] == 1,
    'Rain',
    'Dry'
)

In [29]:
features['rain_status'].value_counts()

rain_status
Dry     18325
Rain     5027
Name: count, dtype: int64

In [30]:
#Temperature–dew point spread
features['temp_dew_spread'] = (
    features['temperature']
    - features['dew_point']
)

In [31]:
features['temp_dew_spread'].describe()

count    23352.000000
mean         5.588442
std          5.398796
min          0.000000
25%          1.400000
50%          4.000000
75%          8.000000
max         40.100000
Name: temp_dew_spread, dtype: float64

In [32]:
#PM2.5 / PM10 ratio
print(
    "Hours where PM2.5 > PM10:",
    (features['pm25'] > features['pm10']).sum()
)

Hours where PM2.5 > PM10: 0


In [33]:
features['pm25_pm10_ratio'] = (
    features['pm25']
    / features['pm10']
)

In [34]:
features[
    'pm25_pm10_ratio'
].describe()

count    23352.000000
mean         0.708807
std          0.099759
min          0.257143
25%          0.679245
50%          0.697297
75%          0.701538
max          1.000000
Name: pm25_pm10_ratio, dtype: float64

In [35]:
features.head()

,pm10,pm25,no2,so2,co,temperature,rain,wind_direction,wind_speed,humidity,pressure,dew_point,year,month,month_name,quarter,day,day_of_year,day_of_week,day_name,hour,week_of_year,is_weekend,day_type,time_of_day,season,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,doy_sin,doy_cos,wind_dir_sin,wind_dir_cos,wind_sector,is_raining,rain_status,temp_dew_spread,pm25_pm10_ratio
time,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2022-09-01 00:00:00,53.1,37.1,29.3,5.0,765,20.6,1.2,243,0.8,95,870.2,19.8,2022,9,September,3,1,244,3,Thursday,0,35,0,Weekday,Night,Monsoon,0.000000,1.000000,0.433884,-0.900969,-0.866025,-0.5,-0.870294,-0.492533,-0.891007,-0.453990,SW,1,Rain,0.8,0.698682
2022-09-01 01:00:00,51.6,36.2,24.5,5.4,728,20.7,8.4,194,1.5,95,870.9,19.8,2022,9,September,3,1,244,3,Thursday,1,35,0,Weekday,Night,Monsoon,0.258819,0.965926,0.433884,-0.900969,-0.866025,-0.5,-0.870294,-0.492533,-0.241922,-0.970296,S,1,Rain,0.9,0.701550
2022-09-01 02:00:00,50.6,35.5,18.8,5.8,679,20.9,9.5,225,1.5,94,871.7,19.9,2022,9,September,3,1,244,3,Thursday,2,35,0,Weekday,Night,Monsoon,0.500000,0.866025,0.433884,-0.900969,-0.866025,-0.5,-0.870294,-0.492533,-0.707107,-0.707107,SW,1,Rain,1.0,0.701581
2022-09-01 03:00:00,49.6,34.8,15.9,6.0,653,21.1,7.2,207,0.8,94,872.4,20.2,2022,9,September,3,1,244,3,Thursday,3,35,0,Weekday,Night,Monsoon,0.707107,0.707107,0.433884,-0.900969,-0.866025,-0.5,-0.870294,-0.492533,-0.453990,-0.891007,SW,1,Rain,0.9,0.701613
2022-09-01 04:00:00,47.1,33.1,13.9,6.1,629,21.4,2.6,45,1.0,93,873.0,20.3,2022,9,September,3,1,244,3,Thursday,4,35,0,Weekday,Night,Monsoon,0.866025,0.500000,0.433884,-0.900969,-0.866025,-0.5,-0.870294,-0.492533,0.707107,0.707107,NE,1,Rain,1.1,0.702760


In [36]:
print(
    "Original variables:",
    df.shape[1]
)

print(
    "Feature dataset variables:",
    features.shape[1]
)

print(
    "Observations:",
    features.shape[0]
)

Original variables: 12
Feature dataset variables: 41
Observations: 23352


In [37]:
#missing values for new values
new_feature_missing = (
    features
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
)

new_feature_missing.head(20)

pm10               0
week_of_year       0
day_type           0
time_of_day        0
season             0
hour_sin           0
hour_cos           0
dow_sin            0
dow_cos            0
month_sin          0
month_cos          0
doy_sin            0
doy_cos            0
wind_dir_sin       0
wind_dir_cos       0
wind_sector        0
is_raining         0
rain_status        0
temp_dew_spread    0
is_weekend         0
dtype: int64

In [38]:
#categorical distributions
for column in [
    'season',
    'day_type',
    'time_of_day',
    'wind_sector',
    'rain_status'
]:

    print(f"\n{column}")
    print(features[column].value_counts())


season
season
Monsoon         6576
Winter          6504
Pre-Monsoon     5880
Post-Monsoon    4392
Name: count, dtype: int64

day_type
day_type
Weekday    16680
Weekend     6672
Name: count, dtype: int64

time_of_day
time_of_day
Night        5838
Morning      5838
Afternoon    5838
Evening      5838
Name: count, dtype: int64

wind_sector
wind_sector
S     11134
NW     4313
N      2339
SE     1894
W      1080
NE      973
SW      810
E       809
Name: count, dtype: int64

rain_status
rain_status
Dry     18325
Rain     5027
Name: count, dtype: int64


In [39]:
#daily analytical dataset
daily_hour_count = (
    features['pm25']
    .resample('D')
    .count()
)

daily_hour_count.value_counts()

pm25
24    973
Name: count, dtype: int64

In [40]:
print(
    "Days without 24 observations:",
    (daily_hour_count != 24).sum()
)

Days without 24 observations: 0


In [41]:
#daily aggregations
daily = features.resample('D').agg(

    pm25_mean=('pm25', 'mean'),
    pm10_mean=('pm10', 'mean'),
    no2_mean=('no2', 'mean'),
    so2_mean=('so2', 'mean'),
    co_mean=('co', 'mean'),

    temperature_mean=('temperature', 'mean'),
    temperature_min=('temperature', 'min'),
    temperature_max=('temperature', 'max'),

    rain_total=('rain', 'sum'),

    wind_speed_mean=('wind_speed', 'mean'),

    humidity_mean=('humidity', 'mean'),

    pressure_mean=('pressure', 'mean'),

    dew_point_mean=('dew_point', 'mean')
)

In [42]:
#wind direction requires circular averagingfeatures['wind_direction'].resample('D').mean()
features['wind_direction'].resample('D').mean()

time
2022-09-01    210.375000
2022-09-02    236.958333
2022-09-03    207.208333
2022-09-04    181.958333
2022-09-05    179.083333
                 ...    
2025-04-26    220.291667
2025-04-27    155.958333
2025-04-28    156.750000
2025-04-29    176.708333
2025-04-30    167.875000
Freq: D, Name: wind_direction, Length: 973, dtype: float64

In [43]:
daily_wind_sin = (
    features['wind_dir_sin']
    .resample('D')
    .mean()
)

daily_wind_cos = (
    features['wind_dir_cos']
    .resample('D')
    .mean()
)

In [44]:
daily['wind_direction_mean'] = (
    np.degrees(
        np.arctan2(
            daily_wind_sin,
            daily_wind_cos
        )
    ) % 360
)

In [45]:
#calendar variables to daily data
daily['year'] = daily.index.year
daily['month'] = daily.index.month
daily['month_name'] = daily.index.month_name()
daily['day_of_week'] = daily.index.dayofweek
daily['day_name'] = daily.index.day_name()

daily['is_weekend'] = (
    daily['day_of_week'] >= 5
).astype(int)

In [46]:
daily['season'] = (
    daily['month']
    .apply(get_season)
)

daily['season'] = pd.Categorical(
    daily['season'],
    categories=season_order,
    ordered=True
)

In [47]:
#AQI interpolation function
def calculate_subindex(
    concentration,
    breakpoints
):
    """
    Convert pollutant concentration to AQI sub-index
    using breakpoint interpolation.
    """

    for (
        c_low,
        c_high,
        i_low,
        i_high
    ) in breakpoints:

        if c_low <= concentration <= c_high:

            aqi = (
                (i_high - i_low)
                / (c_high - c_low)
                * (concentration - c_low)
                + i_low
            )

            # Standard nearest-integer rounding
            return int(
                np.floor(aqi + 0.5)
            )

    return np.nan

In [48]:
#PM2.5 breakpoints
PM25_BREAKPOINTS = [
    (0.0,   9.0,   0,   50),
    (9.1,  35.4,  51,  100),
    (35.5, 55.4, 101,  150),
    (55.5, 125.4, 151, 200),
    (125.5, 225.4, 201, 300),
    (225.5, 325.4, 301, 500)
]

In [49]:
def truncate_pm25(value):

    return (
        np.floor(value * 10)
        / 10
    )

In [50]:
#Daily PM2.5 AQI
daily['pm25_for_aqi'] = (
    daily['pm25_mean']
    .apply(truncate_pm25)
)

In [51]:
daily['pm25_aqi'] = (
    daily['pm25_for_aqi']
    .apply(
        lambda x:
        calculate_subindex(
            x,
            PM25_BREAKPOINTS
        )
    )
)

In [52]:
daily[
    [
        'pm25_mean',
        'pm25_for_aqi',
        'pm25_aqi'
    ]
].head(20)

,pm25_mean,pm25_for_aqi,pm25_aqi
time,,,
2022-09-01,24.866667,24.8,80
2022-09-02,23.312500,23.3,77
2022-09-03,22.125000,22.1,75
2022-09-04,22.654167,22.6,76
2022-09-05,21.600000,21.5,74
2022-09-06,23.083333,23.0,77
2022-09-07,22.612500,22.6,76
2022-09-08,26.262500,26.2,83
2022-09-09,23.437500,23.4,78
